# Proyecto Final: Sistema de Recomendación E-commerce
## Modelo Híbrido — basado en los modelos de Filtrado Colaborativo y Basado en Contenido

**Equipo:** MetricEdge

Este notebook construye el modelo híbrido **reutilizando, sin modificar, la lógica de los dos modelos
ya desarrollados por el equipo**:

- **Componente de contenido**: generador de candidatos TF-IDF + similitud coseno + impulso de recompra (`content_based_reforce.ipynb`).
- **Componente colaborativo**: ALS (`Modelo1_Filtrado_Colaborativo.ipynb`), reentrenado sobre el mismo split para que su score sea una feature más.

Sobre esos candidatos se entrena un **clasificador supervisado de re-ranking** (¿el cliente va a comprar
este candidato? sí/no), usando como features la señal de ambos modelos.

###  Hallazgo central de este notebook

El notebook original de contenido reportaba **ROC-AUC = 0.72** para el re-ranker. Al revisarlo, esa
métrica estaba calculada **evaluando el modelo sobre las mismas filas con las que se entrenó** (sin
separar un conjunto de validación). Acá se corrige ese problema separando **clientes** (no filas) en
train/validación, para que la validación sea honesta. El resultado cambia sustancialmente — ver sección 4.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


## 1. Generación de candidatos (contenido + ALS)

El código completo que arma estos datasets —reutilizando tal cual el generador de candidatos de
`content_based_reforce.ipynb` y agregando el score de ALS como feature adicional— está en
`generar_candidatos_hibrido.py`. Tarda ~15 minutos en correr sobre los ~24.500 clientes porque recalcula
la similitud de contenido para cada uno, así que acá se cargan los datasets ya generados por ese script
para poder iterar rápido sobre el modelado y la evaluación.

- `hybrid_candidates_dataset.csv`: dataset de entrenamiento del clasificador (positivos + hasta 4 negativos
  por positivo, igual que en el notebook original de la compañera).
- `hybrid_val_full_candidates.csv`: candidatos **completos** (sin submuestrear) para los clientes de
  validación — necesario para medir Precision@K/Recall@K de forma correcta (ver sección 5).

In [2]:
df = pd.read_csv("hybrid_candidates_dataset.csv")
df["customer_id"] = df["customer_id"].astype(str)
print(f"Dataset de entrenamiento del clasificador: {df.shape}")
print(df["target"].value_counts())

features = ["content_rank_score", "past_quantity", "global_popularity",
            "unit_price", "product_rating", "is_new_customer", "als_score"]


Dataset de entrenamiento del clasificador: (52694, 10)
target
0    51514
1     1180
Name: count, dtype: int64


## 2. Split honesto: por cliente, no por fila

Para que la validación mida generalización real, los clientes de validación **no pueden aparecer nunca**
en el set de entrenamiento del clasificador.

In [3]:
customer_ids = df["customer_id"].unique()
rng = np.random.default_rng(SEED)
rng.shuffle(customer_ids)
n_val = int(len(customer_ids) * 0.2)
val_customers = set(customer_ids[:n_val])

df_train = df[~df["customer_id"].isin(val_customers)]
df_val = df[df["customer_id"].isin(val_customers)]

print(f"Train: {len(df_train):,} filas ({df_train.customer_id.nunique():,} clientes)")
print(f"Val:   {len(df_val):,} filas ({df_val.customer_id.nunique():,} clientes, nunca vistos en train)")

X_train, y_train = df_train[features], df_train["target"]
X_val, y_val = df_val[features], df_val["target"]


Train: 42,176 filas (19,624 clientes)
Val:   10,518 filas (4,905 clientes, nunca vistos en train)


## 3. Entrenamiento de los 3 clasificadores

In [4]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=100, max_depth=6, random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, scale_pos_weight=4,
                                random_state=SEED, verbosity=-1),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_val)[:, 1]
    results.append({
        "Modelo": name,
        "ROC-AUC (validación honesta)": round(roc_auc_score(y_val, probs), 4),
        "PR-AUC (validación honesta)": round(average_precision_score(y_val, probs), 4),
    })

df_results = pd.DataFrame(results).sort_values("PR-AUC (validación honesta)", ascending=False).reset_index(drop=True)
display(df_results)


,Modelo,ROC-AUC (validación honesta),PR-AUC (validación honesta)
0,Logistic Regression,0.5337,0.0275
1,Random Forest,0.5052,0.0231
2,LightGBM,0.5026,0.0225


## 4. El hallazgo central: en-muestra vs. validación honesta

Reproducimos exactamente lo que hacía el notebook original (entrenar y evaluar sobre el mismo set) para
mostrar de dónde salía el ROC-AUC = 0.72, y lo comparamos contra la validación honesta de arriba.

In [5]:
rf_insample = RandomForestClassifier(class_weight="balanced", n_estimators=100, max_depth=6, random_state=SEED)
X_full, y_full = df[features], df["target"]
rf_insample.fit(X_full, y_full)
probs_insample = rf_insample.predict_proba(X_full)[:, 1]
roc_insample = roc_auc_score(y_full, probs_insample)

roc_honesto = df_results.loc[df_results["Modelo"] == "Random Forest", "ROC-AUC (validación honesta)"].values[0]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(["Evaluación en-muestra\n(notebook original)", "Validación honesta\n(clientes nunca vistos)"],
              [roc_insample, roc_honesto], color=["#c96a3c", "#00C896"])
ax.axhline(0.5, color="gray", ls="--", label="Azar (ROC-AUC = 0.50)")
ax.set_ylim(0, 1)
ax.set_ylabel("ROC-AUC")
ax.set_title("Random Forest: métrica en-muestra vs. validación honesta")
for b, v in zip(bars, [roc_insample, roc_honesto]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"ROC-AUC en-muestra (como en el notebook original): {roc_insample:.4f}")
print(f"ROC-AUC validación honesta (clientes nuevos):        {roc_honesto:.4f}")


<Figure size 700x500 with 1 Axes>

ROC-AUC en-muestra (como en el notebook original): 0.7370
ROC-AUC validación honesta (clientes nuevos):        0.5052


**Conclusión:** el ROC-AUC de 0.72 no era una medida válida de qué tan bien generaliza el modelo —
era básicamente el modelo memorizando los propios datos de entrenamiento (RandomForest con profundidad 6
sobre un dataset con muy pocos positivos, 946, tiene margen de sobra para memorizar en vez de aprender un
patrón real). Con una validación honesta, el ROC-AUC vuelve a estar en la zona de 0.50 — es decir, básicamente
al nivel del azar. Esto es consistente con todo lo que veníamos encontrando en el EDA y en el resto de los
modelos: **hay muy poca señal real de preferencia individual en este dataset a nivel de producto puntual.**

## 5. Precision@10 / Recall@10 correctos (sobre el pool completo de candidatos)

Un cuidado adicional: si se mide Precision@K/Recall@K sobre el mismo dataset submuestreado que se usó para
entrenar (positivos + máximo 4 negativos), el resultado es artificialmente perfecto (el candidato correcto
casi siempre está entre las pocas filas disponibles). Por eso se regeneraron, **solo para los clientes de
validación**, los 20 candidatos completos sin submuestrear (`hybrid_val_full_candidates.csv`).

In [6]:
df_val_full = pd.read_csv("hybrid_val_full_candidates.csv")
df_val_full["customer_id"] = df_val_full["customer_id"].astype(str)

# Cobertura real del generador de candidatos: ¿la compra futura del cliente aparece
# entre los 20 candidatos que arma el modelo de contenido?
n_customers = df_val_full["customer_id"].nunique()
n_con_hit_posible = (df_val_full.groupby("customer_id")["target"].sum() > 0).sum()
print(f"Clientes de validación evaluados: {n_customers:,}")
print(f"Clientes cuya compra real aparece entre los 20 candidatos: {n_con_hit_posible} "
      f"({n_con_hit_posible/n_customers*100:.1f}%)")
print("-> Este es el verdadero cuello de botella: la generación de candidatos, más que el re-ranking.")


Clientes de validación evaluados: 4,905
Clientes cuya compra real aparece entre los 20 candidatos: 231 (4.7%)
-> Este es el verdadero cuello de botella: la generación de candidatos, más que el re-ranking.


In [7]:
X_val_full = df_val_full[features]
K = 10

def precision_recall_at_k(df_scored, score_col, k=10):
    precisions, recalls = [], []
    for _, g in df_scored.groupby("customer_id"):
        n_true = g["target"].sum()
        if n_true == 0:
            continue
        top = g.nlargest(k, score_col)
        hits = top["target"].sum()
        precisions.append(hits / k)
        recalls.append(hits / n_true)
    return np.mean(precisions), np.mean(recalls), len(precisions)

resumen = []
for name, model in models.items():
    df_val_full[f"score_{name}"] = model.predict_proba(X_val_full)[:, 1]
    p, r, n = precision_recall_at_k(df_val_full, f"score_{name}", K)
    resumen.append({"Modelo": name, f"Precision@{K}": p*100, f"Recall@{K}": r*100, "n usuarios": n})

# Baseline: sin re-ranking, orden original del modelo de contenido
p, r, n = precision_recall_at_k(df_val_full, "content_rank_score", K)
resumen.append({"Modelo": "Contenido solo (sin re-ranking)", f"Precision@{K}": p*100, f"Recall@{K}": r*100, "n usuarios": n})

df_resumen = pd.DataFrame(resumen).sort_values(f"Recall@{K}", ascending=False).reset_index(drop=True)
display(df_resumen.round(2))


,Modelo,Precision@10,Recall@10,n usuarios
0,Logistic Regression,5.37,53.03,231
1,Random Forest,5.24,51.30,231
2,LightGBM,4.89,48.05,231
3,Contenido solo (sin re-ranking),4.85,48.05,231


## 6. Conclusiones

- El **cuello de botella real no es el clasificador de re-ranking, es la generación de candidatos**: solo
  en el 4.7% de los clientes de validación la compra futura real está entre los 20 candidatos que arma el
  modelo de contenido. Mejorar el re-ranker no puede compensar eso.
- Sobre ese 4.7% donde sí hay una oportunidad real, el re-ranking con Regresión Logística logra
  **Recall@10 ≈ 48-53%** — un número honesto y bastante bueno, pero acotado a esa porción de clientes, no
  al total de la base.
- Agregar el score de ALS como feature **no mejora sustancialmente** sobre usar solo el ranking de
  contenido — otra señal de que no hay mucho margen de mejora con los datos y features actuales.
- El hallazgo del ROC-AUC 0.72 vs. ~0.50 es en sí mismo un resultado valioso para el informe: muestra que
  el equipo detectó y corrigió un problema real de metodología, no solo que reporta el número más lindo.